#### Smart RAG Pipeline - End-to-End Question Answering System
=========================================================

This notebook demonstrates the complete Smart RAG (Retrieval-Augmented Generation) 
pipeline for intelligent question answering using AWS Bedrock Knowledge Base.

#### Pipeline Architecture:
---------------------------
The system implements a 6-stage pipeline that progressively refines retrieval 
and generation to maximize answer quality:

WORKFLOW =

Stage 1: Query Analysis
- Extract user intent (how-to, conceptual, troubleshooting, etc.)
- Identify mentioned tools/technologies
- Detect specificity level (precise vs. exploratory)
- Output: QueryAnalysis with confidence score

Stage 2: Retrieval Planning
- Select optimal strategy based on query analysis
- Configure metadata filters (page_h1, root_heading)
- Set retrieval budget (top_k, expansion)
- Build 4-step fallback ladder for progressive relaxation

Stage 3: Filter Generation & Refinement
- LLM refines filters semantically (expand synonyms, related terms)
- Validate against KB catalog (available page_h1/root_heading values)
- Translate to AWS Bedrock filter format
- Output: Refined filters with reasoning

Stage 4: Knowledge Base Retrieval
- Execute initial retrieval with refined filters
- Check quality threshold (min_results count)
- If insufficient, trigger fallback ladder:
  - Step 1: Relax secondary filters (remove root_heading)
  - Step 2: Broaden primary filters (expand page_h1 list)
  - Step 3: Remove all filters, increase top_k
  - Step 4: Maximum retrieval (50+ docs, no filters)
- Output: RetrievalResult with documents and metadata

Stage 5: Answer Generation
- Build context from retrieved documents (token-managed)
- Generate answer with inline citations [Doc N]
- Calculate multi-factor confidence score:
  - Top document score (35%)
  - Score consistency (15%)
  - Document coverage (15%)
  - Answer specificity (10%)
  - Refusal detection penalty (15%)
  - Query-answer alignment (10%)
- Output: Answer text + confidence score

Stage 6: Quality Validation
- Check answer length (min 10 words)
- Detect hallucination markers ("as an AI", "I don't have access")
- Verify topic alignment (keyword overlap)
- Count uncertainty markers ("unclear", "not sure", "might be")
- Output: Validation report with issues list


#### Data Flow Example:
-----------------------

User Query: "How do I install RStudio on Ubuntu?"
```
Step 1 - Query Analysis:
  Input: "How do I install RStudio on Ubuntu?"
  Output: QueryAnalysis {
    intent_primary: {"type": "how-to", "confidence": 0.95},
    tools_mentioned: ["RStudio", "Ubuntu"],
    specificity: "precise",
    confidence_score: 0.92
  }

Step 2 - Retrieval Planning:
  Input: QueryAnalysis from Step 1
  Output: RetrievalPlan {
    initial_strategy: "PRECISE",
    filters: {"page_h1": {"equals": "RStudio"}, "root_heading": {"in": ["Install", "Setup"]}},
    budget: {"top_k": 15, "expansion": 1.2},
    fallback: [5-step ladder...]
  }

Step 3 - Filter Refinement:
  Input: RetrievalPlan + kb_catalog
  LLM Reasoning: "Query is specific to RStudio installation. Expand to include 
                  'Getting Started' as it often contains installation steps."
  Output: {
    "final_filters": {
      "page_h1": ["RStudio", "RStudio IDE"],
      "root_heading": ["Install", "Setup", "Getting Started"]
    },
    "reasoning": "Expanded to include RStudio IDE variant and Getting Started section",
    "confidence_in_filters": 0.88
  }

Step 4 - Retrieval Execution:
  Initial Attempt: 
    - Filters: page_h1=["RStudio", "RStudio IDE"], root_heading=[Install, Setup, Getting Started]
    - top_k: 15
    - Retrieved: 12 documents
    - Quality Check: 12 >= min_results (8) ✅ PASS
  
  Output: RetrievalResult {
    documents: [12 docs with scores 0.87-0.72],
    count: 12,
    strategy_used: "PRECISE",
    filters_used: {...},
    fallback_step: 0,
    notes: "Initial strategy succeeded with LLM-refined filters"
  }

Step 5 - Answer Generation:
  Context Built: 
    --- Document 1 ---
    Source: RStudio > Install
    Content: To install RStudio on Ubuntu...
    
    --- Document 2 ---
    Source: RStudio > Setup
    Content: After installation, configure...
    
    [... 10 more documents ...]
  
  Prompt: "Based on retrieved context, answer: How do I install RStudio on Ubuntu?
           Use inline citations [Doc N]..."
  
  Generated Answer:
    "To install RStudio on Ubuntu, follow these steps [Doc 1]:
     1. Update package lists: sudo apt update
     2. Install dependencies: sudo apt install r-base gdebi-core
     3. Download RStudio: wget https://download1.rstudio.org/...
     4. Install: sudo gdebi rstudio-*.deb [Doc 2]
     
     After installation, launch RStudio from Applications menu or terminal [Doc 3]."
  
  Confidence Calculation:
    - top_score: 0.87 → 0.87 * 0.35 = 0.305
    - consistency: low variance → 0.92 * 0.15 = 0.138
    - doc_count: 12/5 = 1.0 → 1.0 * 0.15 = 0.150
    - specificity: 45 words / 120 = 0.375 → 0.375 * 0.10 = 0.038
    - refusal: 0.0 → (1-0) * 0.15 = 0.150
    - alignment: 5/6 terms overlap → 0.833 * 0.10 = 0.083
    Total: 0.864

Step 6 - Validation:
  Checks:
    ✅ Length: 45 words > 10 minimum
    ✅ Hallucination: No "as an AI" phrases detected
    ✅ Topic alignment: 5/6 query terms present
    ✅ Uncertainty: Only 0 markers (< 3 threshold)
  
  Output: {
    "is_valid": true,
    "issues": [],
    "word_count": 45
  }

Final Output - SmartAnswer:
  {
    "answer": "To install RStudio on Ubuntu, follow these steps [Doc 1]:...",
    "sources": [
      {"page_h1": "RStudio", "root_heading": "Install", "score": 0.87},
      {"page_h1": "RStudio", "root_heading": "Setup", "score": 0.84},
      {"page_h1": "RStudio IDE", "root_heading": "Getting Started", "score": 0.79}
    ],
    "retrieval_metadata": {
      "strategy": "PRECISE",
      "filters_used": {"page_h1": ["RStudio", "RStudio IDE"], ...},
      "fallback_step": 0,
      "docs_retrieved": 12,
      "latency_ms": 4523
    },
    "confidence": 0.864,
    "validation_issues": []
  }
```

#### Performance Benchmarks:
---------------------------
Based on typical queries:

- **Latency**:
  - Simple query (no fallback): 4-6 seconds
  - Complex query (1-2 fallbacks): 6-10 seconds
  - Maximum fallback (4 steps): 10-15 seconds

- **Token Usage** (per query):
  - Query analysis: ~300 tokens
  - Retrieval planning: ~500 tokens
  - Filter generation: ~600 tokens
  - Answer generation: ~2000 tokens
  - Total: ~3400 tokens average

- **Cost** (per query):
  - LLM inference: $0.002-0.004
  - KB retrieval: $0.0001-0.0003
  - Total: ~$0.002-0.005 per query

- **Success Rates** (empirical):
  - Initial strategy success: 65-70%
  - Success after 1 fallback: 85-90%
  - Success after 2+ fallbacks: 95%+
  - Complete failure (all fallbacks): <5%


#### Troubleshooting Guide:
--------------------------

- **Issue: Low confidence scores consistently**
  - Check kb_catalog completeness (missing page_h1/headings?)
  - Review retrieval strategy selection logic
  - Inspect document relevance scores in verbose mode
  - Verify KB embedding quality

- **Issue: Frequent fallbacks triggered**
  - Filters may be too restrictive
  - min_results threshold may be too high
  - KB may lack coverage for query topics
  - Consider adjusting fallback ladder steps

- **Issue: Hallucination in answers**
  - Check SYSTEM_PROMPT instructions
  - Review context building (token truncation?)
  - Verify document content quality in KB
  - Adjust temperature (currently 0.1)

- **Issue: Slow performance**
  - Enable caching for repeated queries
  - Optimize max_context_tokens (default 6000)
  - Consider async/parallel fallback evaluation
  - Review AWS region latency

- **Issue: Validation failures**
  - Inspect validation_issues for specific problems
  - Check if answer length thresholds are appropriate
  - Review uncertainty marker patterns
  - Verify query-answer alignment logic


In [ ]:
# Import libraries and paths
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

# Add project root to Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from config import KB_ID, MODEL_ID, REGION

# Import the QueryAnalyser and related components
from helpers.apug.rag.query_analyser_07_01 import QueryAnalyser
from helpers.apug.rag.retrieval_planner_07_02 import RetrievalPlanner
from helpers.apug.rag.filter_generator_07_03 import FilterGenerator
from helpers.apug.rag.ask_smart_07_04 import AskSmart,SmartAnswer

In [ ]:
# ============================================================================
# HELPER FUNCTION
# ============================================================================
def print_result(result):
    """Safely print SmartAnswer with error handling."""
    print(f"\nANSWER:\n{result.answer}\n")
    print(f"Confidence: {result.confidence:.0%}")
    print(f"Sources: {len(result.sources)} documents")
    
    meta = result.retrieval_metadata
    print(f"Strategy: {meta.get('strategy', 'N/A')}")
    print(f"Fallback steps: {meta.get('fallback_step', 'N/A')}")
    print(f"Docs retrieved: {meta.get('docs_retrieved', 0)}")
    print(f"Latency: {meta.get('latency_ms', 0):.0f}ms")
    
    if 'error' in meta:
        print(f"\n⚠️ ERROR: {meta.get('error_type', 'Unknown')}: {meta.get('error', 'Unknown error')}")
    
    print("="*80)
    
    if result.validation_issues:
        print(f"\n⚠️ Validation issues: {', '.join(result.validation_issues)}")
    
    if result.sources:
        print("\n" + "="*70)
        print("SOURCES:")
        print("="*70)
        for i, source in enumerate(result.sources, 1):
            print(f"\n{i}. {source['page_h1']} > {source['root_heading']}")
            print(f"   Score: {source['score']:.3f}")
            print(f"   {source['content'][:150]}...")
    else:
        print("\n⚠️ No sources returned")

In [ ]:

# ============================================================================
# STEP 1: KNOWLEDGE BASE CATALOG
# ============================================================================
# Knowledge base catalog
kb_catalog_json_path = "../data/kb_catalog.json"
with open(kb_catalog_json_path, "r") as f:
    kb_catalog = json.load(f)

# ============================================================================
# STEP 2: CALLING ALL THE FUNCTIONS
# ============================================================================
# Initialize components
query_analyser = QueryAnalyser(model_id=MODEL_ID, region=REGION)
retrieval_planner = RetrievalPlanner(min_results=3)
filter_generator = FilterGenerator(kb_id=KB_ID, region=REGION, llm_model_id=MODEL_ID)
ask_smart = AskSmart(
    analyser=query_analyser,
    planner=retrieval_planner,
    filter_gen=filter_generator,
    kb_id=KB_ID,
    kb_catalog=kb_catalog,
    answer_model_id=MODEL_ID,
    region=REGION,
    max_context_tokens=6000
)

In [ ]:
# ============================================================================
# STEP 3: FIRST QUESTION
# ============================================================================
print("Initializing AI Assistant...\n")

# Example Query
query_1 = "On the topic of the data uploader. Is there a process to request the removal of an existing table? " \
    "I have 2 tables that got data ingested and are now in Athena. But those tables changed during the latest designs review and now they are obsolete. " \
    "How can I get those deleted? What is the process to request such action? " \
    "Worth to highlight that the data stored in those tables is replaceable and there is no need to keep an archive of it"

print(f"QUESTION:\n{query_1}\n")
print("="*80)

# Execute query with verbose to see what's happening
result_1 = ask_smart.ask(query_1, verbose=True)

# Use the helper function instead of manual printing
print_result(result_1)

In [ ]:
# ============================================================================
# SECOND QUESTION
# ============================================================================

question9 = "Good morning, all - I am unable to log into R Studio today (v4.5.1-1). I keep getting an error message that says" \
    "502 Bad Gateway" \
    "or sometimes" \
    "504 Gateway Time-out." \
        "Nothing i try is resolving this problem: restarting the laptop; refreshing the webpage; clearing the browser cache." \
            " Some helpful advice would be much appreciated!"


print(f"QUESTION:\n{question9}\n")
print("="*80)

result_9 = ask_smart.ask(question9, verbose=False)

# Use the helper function instead of manual printing
print_result(result_9)

In [ ]:
# ============================================================================
# THIRD QUESTION
# ============================================================================

question10 = " QuickSight question: I have changed the schema of my data in Athena. This doesn't automatically propagate to QuickSight." \
    " Is there a way of updating the schema of QS datasets, without having to delete and readd datasets? We do not use SPICE yet." \
    "My understanding is that a view is taken of the data, so perhaps the data source configuration can be kicked for our views to reload the schema?"

print(f"QUESTION:\n{question10}\n")
print("="*80)

result_3 = ask_smart.ask(question10, verbose=False)

print_result(result_3)


In [ ]:
# ============================================================================
# DETAILED INSPECTION OF INDIVIDUAL QUESTIONS(Uncomment to see technical details)
# ============================================================================
""" 


print("\n" + "="*80)
print("DETAILED TECHNICAL ANALYSIS")
print("="*80)

def print_detailed_result(result: SmartAnswer, query: str):
    """
    #Helper function to print comprehensive result details
"""
    print(f"\nQuery: {query}")
    print(f"\nAnswer ({len(result.answer.split())} words):")
    print(result.answer)
    print(f"\nConfidence Breakdown:")
    print(f"  Overall: {result.confidence:.3f}")
     
    print(f"\nRetrieval Metadata:")
    metadata = result.retrieval_metadata
    print(f"  KB ID: {metadata['kb_id']}")
    print(f"  Strategy: {metadata['strategy']}")
    print(f"  Filters Used: {json.dumps(metadata['filters_used'], indent=4)}")
    print(f"  Fallback Step: {metadata['fallback_step']}")
    print(f"  Documents Retrieved: {metadata['docs_retrieved']}")
    print(f"  Query Type: {metadata['query_type']}")
    print(f"  Tools Mentioned: {metadata['tools_mentioned']}")
    print(f"  Analyzer Confidence: {metadata['analyzer_confidence']:.2f}")
    print(f"  Latency: {metadata['latency_ms']:.0f}ms")
     
    print(f"\nSources (Top 3):")
    for i, source in enumerate(result.sources[:3], 1):
        print(f"\n  Source {i}:")
        print(f"    Page: {source['page_h1']}")
        print(f"    Section: {source['root_heading']}")
        print(f"    Relevance Score: {source['score']:.3f}")
        print(f"    Preview: {source['content'][:150]}...")
     
    print(f"\nValidation:")
    if result.validation_issues:
        print(f"  Issues Detected:")
        for issue in result.validation_issues:
            print(f"    - {issue}")
    else:
        print(f"  ✓ All checks passed")
    
    print("\n" + "─"*80)

print_detailed_result(result_1, query_1)

# ============================================================================
# PERFORMANCE MONITORING (Uncomment for multi-query analysis)
# ============================================================================

def analyze_performance(results: List[SmartAnswer]):
    """
    # Analyze performance metrics across multiple queries
"""
    total_queries = len(results)
     
    avg_confidence = sum(r.confidence for r in results) / total_queries
    avg_latency = sum(r.retrieval_metadata['latency_ms'] for r in results) / total_queries
    avg_docs = sum(r.retrieval_metadata['docs_retrieved'] for r in results) / total_queries
     
    no_fallback = sum(1 for r in results if r.retrieval_metadata['fallback_step'] == 0)
    with_fallback = total_queries - no_fallback
    with_issues = sum(1 for r in results if r.validation_issues)
    
    print(f"\nPerformance Summary ({total_queries} queries):")
    print(f"\n  Confidence:")
    print(f"    Average: {avg_confidence:.3f}")
    print(f"    Min: {min(r.confidence for r in results):.3f}")
    print(f"    Max: {max(r.confidence for r in results):.3f}")
    
    print(f"\n  Latency:")
    print(f"    Average: {avg_latency:.0f}ms")
    print(f"    Min: {min(r.retrieval_metadata['latency_ms'] for r in results):.0f}ms")
    print(f"    Max: {max(r.retrieval_metadata['latency_ms'] for r in results):.0f}ms")
    
    print(f"\n  Retrieval:")
    print(f"    Avg Documents: {avg_docs:.1f}")
    print(f"    No Fallback: {no_fallback}/{total_queries} ({no_fallback/total_queries*100:.1f}%)")
    print(f"    With Fallback: {with_fallback}/{total_queries} ({with_fallback/total_queries*100:.1f}%)")
    
    print(f"\n  Quality:")
    print(f"    With Validation Issues: {with_issues}/{total_queries} ({with_issues/total_queries*100:.1f}%)")
    print(f"    Passed Validation: {total_queries-with_issues}/{total_queries} ({(total_queries-with_issues)/total_queries*100:.1f}%)")

analyze_performance([result_1])

# ============================================================================
# INTERACTIVE MODE (Uncomment for live testing)
# ============================================================================

print("\n" + "="*80)
print("INTERACTIVE MODE")
print("="*80)
print("\nTry your own queries:")
print('result = ask_smart.ask("Your question here", verbose=False)')
print("print(result.answer)")
"""

In [ ]:
""" 
If you are intrested in the futher understanding of each function. you can use the below code
#---------------------------
# Test Query analyser function
#---------------------------
analyser = QueryAnalyser()
analysis = analyser.analyse("How do I set up Airflow and schedule a DAG?", verbose=True)
print(analysis)

analysis.intent_primary  # you can try different definations such as intent_primary, _parse_analysis,...(check definition of more)

#---------------------------
# Test retrieval planner
#---------------------------
planner = RetrievalPlanner(min_results=3)
# Optionally load kb_catalog.json here and pass to plan()
plan = planner.plan(analysis, kb_catalog=kb_catalog)

print(plan)

print(f"Strategy: {plan.initial_strategy}")
print(f"Filters: {plan.filters}")
print(f"Budget: {plan.budget.top_k} docs, expansion={plan.budget.expansion}")
print(f"Fallback steps: {len(plan.fallback.ladder)}")


# Test filter genration
#---------------------------
filter_gen = FilterGenerator(
    kb_id=KB_ID,
    region=REGION,
    llm_model_id=MODEL_ID,
   
)

# Execute with LLM refinement
result = filter_gen.generate_and_retrieve(
    query="How do I delete a table?",
    plan=plan,
    kb_catalog=kb_catalog,
    verbose=True
)

print(f"Retrieved {result.count} documents using {result.strategy_used}")
print(f"Filters used: {result.filters_used}")

"""